# 04 — Index hybride multilingue

On combine les embeddings multilingues, efficaces pour le sens, avec BM25, utile pour les numéros d'articles et les termes juridiques exacts.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA = PROJECT_ROOT / "data"
RAW = DATA / "raw"
PROCESSED = DATA / "processed"
INDEX = DATA / "index"
for directory in (RAW, PROCESSED, INDEX):
    directory.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT

In [ ]:
import json
import pickle
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

chunks_path = PROCESSED / "chunks.jsonl"
chunks = []
if chunks_path.exists():
    chunks = [json.loads(line) for line in chunks_path.read_text(encoding="utf-8").splitlines()]

texts = [row["text_normalized"] for row in chunks]
model_name = "intfloat/multilingual-e5-base"
model = SentenceTransformer(model_name)
embeddings = model.encode(
    [f"passage: {text}" for text in texts],
    normalize_embeddings=True,
    show_progress_bar=True,
) if texts else np.empty((0, 768), dtype="float32")

np.save(INDEX / "embeddings.npy", embeddings)
(INDEX / "chunks.json").write_text(json.dumps(chunks, ensure_ascii=False), encoding="utf-8")
bm25 = BM25Okapi([text.lower().split() for text in texts]) if texts else None
with (INDEX / "bm25.pkl").open("wb") as stream:
    pickle.dump(bm25, stream)
{"model": model_name, "documents": len(texts), "shape": embeddings.shape}